In [12]:
import pandas as pd

matches_df = pd.read_csv("../data/processed/matches.csv")
deliveries_df = pd.read_csv("../data/processed/deliveries.csv")

matches_df["date"] = pd.to_datetime(matches_df["date"])

print(matches_df.shape, deliveries_df.shape)

(1243, 14) (295732, 12)


In [13]:
def get_phase(over):
    if over < 6:
        return "powerplay"
    elif over < 16:
        return "middle"
    else:
        return "death"

deliveries_df["phase"] = deliveries_df["over"].apply(get_phase)

phase_runs = (
    deliveries_df
    .groupby(["match_id", "innings", "phase"])["runs_total"]
    .sum()
    .reset_index()
)

phase_runs.head()

,match_id,innings,phase,runs_total
0,335982,1,death,64
1,335982,1,middle,97
2,335982,1,powerplay,61
3,335982,2,middle,56
4,335982,2,powerplay,26


In [14]:
phase_runs_wide = phase_runs.pivot_table(
    index=["match_id", "innings"],
    columns="phase",
    values="runs_total",
    fill_value=0
).reset_index()

phase_runs_wide.columns.name = None  # cleans up a leftover label from pivoting
phase_runs_wide.head()

,match_id,innings,death,middle,powerplay
0,335982,1,64.0,97.0,61.0
1,335982,2,0.0,56.0,26.0
2,335983,1,71.0,116.0,53.0
3,335983,2,31.0,113.0,63.0
4,335984,1,23.0,66.0,40.0


In [15]:
inn1 = phase_runs_wide[phase_runs_wide["innings"] == 1].drop(columns="innings")
inn1 = inn1.rename(columns={"powerplay": "pp_runs_inn1", "middle": "mid_runs_inn1", "death": "death_runs_inn1"})

inn2 = phase_runs_wide[phase_runs_wide["innings"] == 2].drop(columns="innings")
inn2 = inn2.rename(columns={"powerplay": "pp_runs_inn2", "middle": "mid_runs_inn2", "death": "death_runs_inn2"})

matches_df = matches_df.merge(inn1, on="match_id", how="left")
matches_df = matches_df.merge(inn2, on="match_id", how="left")

matches_df.head()

,match_id,date,city,venue,team1,team2,toss_winner,toss_decision,winner,result,bat_first_team,bat_second_team,season,has_impact_player_rule,death_runs_inn1,mid_runs_inn1,pp_runs_inn1,death_runs_inn2,mid_runs_inn2,pp_runs_inn2
0,335982,2008-04-18,Bangalore,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,field,Kolkata Knight Riders,normal,Kolkata Knight Riders,Royal Challengers Bangalore,2008,False,64.0,97.0,61.0,0.0,56.0,26.0
1,335983,2008-04-19,Chandigarh,"Punjab Cricket Association Stadium, Mohali",Kings XI Punjab,Chennai Super Kings,Chennai Super Kings,bat,Chennai Super Kings,normal,Chennai Super Kings,Kings XI Punjab,2008,False,71.0,116.0,53.0,31.0,113.0,63.0
2,335984,2008-04-19,Delhi,Feroz Shah Kotla,Delhi Daredevils,Rajasthan Royals,Rajasthan Royals,bat,Delhi Daredevils,normal,Rajasthan Royals,Delhi Daredevils,2008,False,23.0,66.0,40.0,0.0,77.0,55.0
3,335986,2008-04-20,Kolkata,Eden Gardens,Kolkata Knight Riders,Deccan Chargers,Deccan Chargers,bat,Kolkata Knight Riders,normal,Deccan Chargers,Kolkata Knight Riders,2008,False,28.0,43.0,39.0,25.0,61.0,26.0
4,335985,2008-04-20,Mumbai,Wankhede Stadium,Mumbai Indians,Royal Challengers Bangalore,Mumbai Indians,bat,Royal Challengers Bangalore,normal,Mumbai Indians,Royal Challengers Bangalore,2008,False,47.0,71.0,47.0,40.0,86.0,40.0


In [16]:
matches_df["season"] = matches_df["date"].dt.year
matches_df["has_impact_player_rule"] = matches_df["season"] >= 2023

matches_df[["season", "has_impact_player_rule"]].drop_duplicates().sort_values("season")

,season,has_impact_player_rule
0,2008,False
58,2009,False
115,2010,False
175,2011,False
248,2012,False
322,2013,False
398,2014,False
458,2015,False
517,2016,False
577,2017,False


In [17]:
matches_df.columns.tolist()


['match_id',
 'date',
 'city',
 'venue',
 'team1',
 'team2',
 'toss_winner',
 'toss_decision',
 'winner',
 'result',
 'bat_first_team',
 'bat_second_team',
 'season',
 'has_impact_player_rule',
 'death_runs_inn1',
 'mid_runs_inn1',
 'pp_runs_inn1',
 'death_runs_inn2',
 'mid_runs_inn2',
 'pp_runs_inn2']

In [18]:
matches_df = pd.read_csv("../data/processed/matches.csv")
matches_df["date"] = pd.to_datetime(matches_df["date"])

matches_df.columns.tolist()

['match_id',
 'date',
 'city',
 'venue',
 'team1',
 'team2',
 'toss_winner',
 'toss_decision',
 'winner',
 'result',
 'bat_first_team',
 'bat_second_team',
 'season',
 'has_impact_player_rule']

In [19]:
inn1 = phase_runs_wide[phase_runs_wide["innings"] == 1].drop(columns="innings")
inn1 = inn1.rename(columns={"powerplay": "pp_runs_inn1", "middle": "mid_runs_inn1", "death": "death_runs_inn1"})

inn2 = phase_runs_wide[phase_runs_wide["innings"] == 2].drop(columns="innings")
inn2 = inn2.rename(columns={"powerplay": "pp_runs_inn2", "middle": "mid_runs_inn2", "death": "death_runs_inn2"})

matches_df = matches_df.merge(inn1, on="match_id", how="left")
matches_df = matches_df.merge(inn2, on="match_id", how="left")

matches_df["season"] = matches_df["date"].dt.year
matches_df["has_impact_player_rule"] = matches_df["season"] >= 2023

matches_df.columns.tolist()

['match_id',
 'date',
 'city',
 'venue',
 'team1',
 'team2',
 'toss_winner',
 'toss_decision',
 'winner',
 'result',
 'bat_first_team',
 'bat_second_team',
 'season',
 'has_impact_player_rule',
 'death_runs_inn1',
 'mid_runs_inn1',
 'pp_runs_inn1',
 'death_runs_inn2',
 'mid_runs_inn2',
 'pp_runs_inn2']

In [20]:
deliveries_df["is_four"] = deliveries_df["runs_batter"] == 4
deliveries_df["is_six"] = deliveries_df["runs_batter"] == 6
deliveries_df["is_dot"] = deliveries_df["runs_total"] == 0

innings_summary = (
    deliveries_df
    .groupby(["match_id", "innings"])
    .agg(
        total_runs=("runs_total", "sum"),
        total_balls=("ball", "count"),
        fours=("is_four", "sum"),
        sixes=("is_six", "sum"),
        dots=("is_dot", "sum"),
        wickets=("is_wicket", "sum"),
    )
    .reset_index()
)

innings_summary["dot_pct"] = (innings_summary["dots"] / innings_summary["total_balls"] * 100).round(2)
innings_summary["boundary_pct"] = ((innings_summary["fours"] + innings_summary["sixes"]) / innings_summary["total_balls"] * 100).round(2)
innings_summary["run_rate"] = (innings_summary["total_runs"] / innings_summary["total_balls"] * 6).round(2)

innings_summary.head()

,match_id,innings,total_runs,total_balls,fours,sixes,dots,wickets,dot_pct,boundary_pct,run_rate
0,335982,1,222,124,15,14,36,3,29.03,23.39,10.74
1,335982,2,82,101,3,3,50,10,49.50,5.94,4.87
2,335983,1,240,124,20,16,34,5,27.42,29.03,11.61
3,335983,2,207,124,18,9,23,4,18.55,21.77,10.02
4,335984,1,129,122,14,3,54,8,44.26,13.93,6.34


In [21]:
inn1_summary = innings_summary[innings_summary["innings"] == 1].drop(columns="innings")
inn1_summary = inn1_summary.rename(columns={
    "total_runs": "total_runs_inn1", "total_balls": "total_balls_inn1",
    "fours": "fours_inn1", "sixes": "sixes_inn1", "dots": "dots_inn1",
    "wickets": "wickets_inn1", "dot_pct": "dot_pct_inn1",
    "boundary_pct": "boundary_pct_inn1", "run_rate": "run_rate_inn1"
})

inn2_summary = innings_summary[innings_summary["innings"] == 2].drop(columns="innings")
inn2_summary = inn2_summary.rename(columns={
    "total_runs": "total_runs_inn2", "total_balls": "total_balls_inn2",
    "fours": "fours_inn2", "sixes": "sixes_inn2", "dots": "dots_inn2",
    "wickets": "wickets_inn2", "dot_pct": "dot_pct_inn2",
    "boundary_pct": "boundary_pct_inn2", "run_rate": "run_rate_inn2"
})

matches_df = matches_df.merge(inn1_summary, on="match_id", how="left")
matches_df = matches_df.merge(inn2_summary, on="match_id", how="left")

matches_df.columns.tolist()

['match_id',
 'date',
 'city',
 'venue',
 'team1',
 'team2',
 'toss_winner',
 'toss_decision',
 'winner',
 'result',
 'bat_first_team',
 'bat_second_team',
 'season',
 'has_impact_player_rule',
 'death_runs_inn1',
 'mid_runs_inn1',
 'pp_runs_inn1',
 'death_runs_inn2',
 'mid_runs_inn2',
 'pp_runs_inn2',
 'total_runs_inn1',
 'total_balls_inn1',
 'fours_inn1',
 'sixes_inn1',
 'dots_inn1',
 'wickets_inn1',
 'dot_pct_inn1',
 'boundary_pct_inn1',
 'run_rate_inn1',
 'total_runs_inn2',
 'total_balls_inn2',
 'fours_inn2',
 'sixes_inn2',
 'dots_inn2',
 'wickets_inn2',
 'dot_pct_inn2',
 'boundary_pct_inn2',
 'run_rate_inn2']

In [22]:
matches_df[["total_runs_inn1", "pp_runs_inn1", "mid_runs_inn1", "death_runs_inn1"]].head()


,total_runs_inn1,pp_runs_inn1,mid_runs_inn1,death_runs_inn1
0,222,61.0,97.0,64.0
1,240,53.0,116.0,71.0
2,129,40.0,66.0,23.0
3,110,39.0,43.0,28.0
4,165,47.0,71.0,47.0


In [23]:
check = matches_df["pp_runs_inn1"] + matches_df["mid_runs_inn1"] + matches_df["death_runs_inn1"] - matches_df["total_runs_inn1"]
check.value_counts()

0.0    1243
Name: count, dtype: int64

In [24]:
matches_df.to_csv("../data/processed/matches.csv", index=False)
print("Saved! Shape:", matches_df.shape)

Saved! Shape: (1243, 38)
